# 조건과 반복

이 노트북에서 다루는 것:
1. 조건에 따른 반복 처리 (단순 루프)
2. `recursion_limit` 와 `GraphRecursionError`
3. 더 복잡한 노드/엣지 연결의 반복
4. 사용자 입력에 따른 반복 조건 설정

## 1. 조건에 따른 반복 처리하기

```
START → a ─┬─→ END     (조건 만족: 길이 >= 7)
           └─→ b → a    (조건 미만족: 다시 a 로)
```

라우팅 함수가 다음 노드 이름을 반환하거나 `END` 를 반환해 종료한다.

In [ ]:
import operator
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    aggregate: Annotated[list, operator.add]

In [ ]:
def a(state: State):
    print(f'Node A 처리 중 현재 상태값 : {state["aggregate"]}')
    return {"aggregate": ["A"]}

def b(state: State):
    print(f'Node B 처리 중 현재 상태값 : {state["aggregate"]}')
    return {"aggregate": ["B"]}

graph_builder = StateGraph(State)
graph_builder.add_node(a)
graph_builder.add_node(b)

In [ ]:
def route(state: State):
    if len(state["aggregate"]) < 7:
        return "b"
    else:
        return END

graph_builder.add_edge(START, "a")
graph_builder.add_conditional_edges("a", route)
graph_builder.add_edge("b", "a")   # b -> a (루프)
graph = graph_builder.compile()

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print(graph.get_graph().draw_mermaid())

In [ ]:
graph.invoke({"aggregate": []})

## 2. `recursion_limit` — 무한 루프 방어

LangGraph 는 노드 실행 횟수 상한을 둔다. 초과하면 `GraphRecursionError` 가 발생한다. `config={"recursion_limit": N}` 으로 호출 단위 조정이 가능하다. 종료 조건에 도달할 수 없는 버그를 빠르게 잡는 안전장치이기도 하다.

In [ ]:
from langgraph.errors import GraphRecursionError

try:
    graph.invoke({"aggregate": []}, config={"recursion_limit": 4})
except GraphRecursionError:   # 반복 종료 조건에 도달할 수 없는 경우
    print("Recursion Error")

## 3. 더 복잡한 노드와 엣지 연결의 반복

```
                ┌── c ──┐
START → a ─→ b ─┤        ├─→ a   (조건 미만족 시 다시)
         │      └── d ──┘
         └─→ END  (조건 만족)
```

- `add_edge(["c", "d"], "a")` : 여러 노드가 모두 끝난 뒤 한 노드로 합류
- 라우팅 함수의 반환 타입을 `Literal["b", END]` 로 힌트하면 가능한 분기를 명시할 수 있다

In [ ]:
import operator
from typing import Annotated, Literal
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    aggregate: Annotated[list, operator.add]

In [ ]:
def a(state: State):
    print(f'Node A 처리 중 현재 상태값 : {state["aggregate"]}')
    return {"aggregate": ["A"]}

def b(state: State):
    print(f'Node B 처리 중 현재 상태값 : {state["aggregate"]}')
    return {"aggregate": ["B"]}

def c(state: State):
    print(f'Node C 처리 중 현재 상태값 : {state["aggregate"]}')
    return {"aggregate": ["C"]}

def d(state: State):
    print(f'Node D 처리 중 현재 상태값 : {state["aggregate"]}')
    return {"aggregate": ["D"]}

graph_builder = StateGraph(State)
graph_builder.add_node(a)
graph_builder.add_node(b)
graph_builder.add_node(c)
graph_builder.add_node(d)

In [ ]:
def route(state: State) -> Literal["b", "__end__"]:
    if len(state["aggregate"]) < 7:
        return "b"
    else:
        return END

graph_builder.add_edge(START, "a")
graph_builder.add_conditional_edges("a", route)
graph_builder.add_edge("b", "c")
graph_builder.add_edge("b", "d")
graph_builder.add_edge(["c", "d"], "a")   # c, d 모두 끝나야 a 재실행
graph = graph_builder.compile()

In [ ]:
result = graph.invoke({"aggregate": []})
print(result)

## 4. 사용자 입력에 따른 반복 조건 설정하기

노드 안에서 `input()` 을 호출해 사람과 대화하며 종료 조건을 판단할 수 있다. 여기서는 사용자 입력에 "반복" 이라는 단어가 있으면 한 번 더, 아니면 종료한다.

또한 메시지를 종류별로 따로 누적할 수 있다 — `human_messages` 와 `ai_messages` 를 분리한다(각각 `add_messages`).

> **실행 주의**: 이 셀은 `input()` 을 호출하므로, 셀 실행 후 나타나는 입력란에 텍스트를 넣어야 한다. "반복" 이라고 입력하면 한 번 더, 다른 텍스트면 종료한다.

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.graph.message import add_messages

class State(TypedDict):
    human_messages: Annotated[list[HumanMessage], add_messages]
    ai_messages: Annotated[list[AIMessage], add_messages]
    retry_num: int

In [ ]:
def chatbot(state: State):
    retry_num = state["retry_num"]
    user_input = input(f"(현재 {retry_num}번째 답변) 사용자 입력: ")
    ai_message = AIMessage(f"{retry_num}번째 답변중!")
    return {"human_messages": [HumanMessage(content=user_input)], "ai_messages": [ai_message]}

def retry(state: State):
    return {"retry_num": state["retry_num"] + 1}

graph_builder = StateGraph(State)
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("retry", retry)

In [ ]:
def route(state: State):
    if "반복" in state["human_messages"][-1].content:
        return "retry"
    else:
        return END

graph_builder.add_edge(START, "chatbot")
graph_builder.add_conditional_edges("chatbot", route)
graph_builder.add_edge("retry", "chatbot")
graph = graph_builder.compile()

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print(graph.get_graph().draw_mermaid())

In [ ]:
# 실행하면 입력란이 나타난다. '반복' 을 입력하면 루프가 한 번 더 돈다.
graph.invoke({"human_messages": [], "ai_messages": [], "retry_num": 0})

## 정리

- **루프** = 조건부 엣지가 이전(또는 자기) 노드로 다시 향하면 됨
- **`recursion_limit`** = 무한 루프 방어. 초과 시 `GraphRecursionError`
- **`add_edge([n1, n2], target)`** = 다중 노드 합류 (모두 끝나야 target 실행)
- **`Literal[...]`** 라우터 반환 타입 힌트로 가능한 분기를 표현
- **사용자 입력 루프** = 노드 안 `input()` + 메시지 누적 + 라우터 조합

여기까지가 LangGraph 그래프 메커니즘의 기초다. 다음 단계에서는 노드 안에 실제 LLM 호출을 넣어 본격적인 에이전트를 만든다 (이때부터 API 키 필요).